# 1. Saved-checkpoint evaluation and submission

Evaluate a complete saved checkpoint without retraining, or generate a Kaggle-format CSV. Select E0, E1 or E2 below; E2 is the final model. Inputs are the checkpoint, saved split/mapping and external competition dataset. All operations that process the full image partitions are disabled by default. New results use a separate output directory, protecting original evidence.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
import torch

# Set this explicitly on Kaggle to the extracted project folder.
PROJECT_ROOT = None
if PROJECT_ROOT is None:
    PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                         if (p / "src/models.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Set PROJECT_ROOT to the extracted project containing src/ and data/processed/")
PROJECT_ROOT = Path(PROJECT_ROOT).resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from src.project import RUNS, load_metadata, read_strings, find_image_directory, find_csv
DATA_ROOT = (Path("/kaggle/input/datasets/leocaoprojects/enge707-2026-s2")
             if Path("/kaggle").exists() else PROJECT_ROOT / "data/raw")
OUTPUT_ROOT = (Path("/kaggle/working/reproduction") if Path("/kaggle").exists()
               else PROJECT_ROOT / "outputs/reproduction")
TRAIN_IMAGE_DIRECTORY = None  # Optional existing cache or explicit train/train directory.
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
RUN_TRAINING = False
RUN_TEST_PREDICTION = False
RUN_EVALUATION = False
split, mapping = load_metadata(PROJECT_ROOT)
print("Project:", PROJECT_ROOT, "Device:", DEVICE)
print(split["split"].value_counts().to_dict())

In [ ]:
from src.evaluation import evaluate_checkpoint, predict_test
EXPERIMENT = "E2"
CHECKPOINT = PROJECT_ROOT / "outputs/models" / (RUNS[EXPERIMENT] + ".pt")
EVALUATION_DIRECTORY = OUTPUT_ROOT / (EXPERIMENT + "_evaluation")
PREDICTION_DIRECTORY = OUTPUT_ROOT / (EXPERIMENT + "_submission")
assert CHECKPOINT.is_file(), f"Missing checkpoint: {CHECKPOINT}"
results = []
for experiment, run in RUNS.items():
    saved = json.loads((PROJECT_ROOT / "outputs/metrics" / run / "validation_metrics.json").read_text())
    values = saved.get("metrics", saved)
    results.append({"experiment": experiment, **{k: values.get(k, saved.get(k)) for k in
                    ["accuracy", "cross_entropy", "macro_f1", "weighted_f1"]}})
display(pd.DataFrame(results))

## 2. Validation

Construct the architecture with `weights=None` and strictly restore all saved weights. Check mapping, split checksum and experiment identity before inference. Use evaluation mode, inference mode, sequential batches and unchanged deterministic preprocessing. Verify all validation IDs, labels and supports. Save sample-weighted loss, classification metrics, predictions and two-panel confusion matrices. Compare the restored result with recorded history using a 1e-6 loss tolerance; discrepancies are reported rather than adjusted.

In [ ]:
if RUN_EVALUATION:
    directory = TRAIN_IMAGE_DIRECTORY or find_image_directory(DATA_ROOT, split["image_id"])
    evaluation = evaluate_checkpoint(PROJECT_ROOT, directory, CHECKPOINT, EXPERIMENT, EVALUATION_DIRECTORY, DEVICE)
    print(evaluation)
else:
    print("Full validation disabled; original reports remain in outputs/metrics/.")

## 3. Test prediction

Sample-submission classes are placeholders, never labels. Preserve all 29,443 original string IDs and row order; zero-pad to five digits only for filename lookup. Images load on demand with no augmentation or dataset copying. The complete checkpoint supplies pretrained features without downloading weights. Input loading can dominate runtime (the original E1 prediction took about 39 minutes).

The saved final CSV is `outputs/submissions/submission_E2.csv`. A new CSV and its verification report are written only when explicitly enabled below. No competition upload is performed. Test accuracy is unknown; reported Kaggle scores are separate from validation metrics.

In [ ]:
if RUN_TEST_PREDICTION:
    prediction_check = predict_test(PROJECT_ROOT, DATA_ROOT, CHECKPOINT, EXPERIMENT, PREDICTION_DIRECTORY, DEVICE)
    print(prediction_check)
else:
    print("Test prediction disabled; use the preserved submission_E2.csv if no new inference is required.")